In [0]:
CATALOG = "workspace"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{GOLD_SCHEMA}"
)

print(f"Catálogo: {CATALOG}")
print(f"Silver:   {SILVER_SCHEMA}")
print(f"Gold:     {GOLD_SCHEMA}")

Catálogo: workspace
Silver:   silver
Gold:     gold


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_info = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.tb_info_filmes"
)

df_financeiro = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.tb_financeiro_filmes"
)

df_metricas = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.tb_metricas_engajamento"
)

df_avaliacoes = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.tb_avaliacoes_usuarios"
)

df_generos = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.tb_generos"
)

df_pessoas_empresas = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.tb_pessoas_empresas"
)

print("[OK] Tabelas Silver carregadas.")

[OK] Tabelas Silver carregadas.


In [0]:
df_dim_movies_base = (
    df_info
    .select(
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
)

In [0]:
window_sk_movies = (
    Window.orderBy("id_filme")
)

df_dim_movies = (
    df_dim_movies_base
    .withColumn(
        "sk_movie_id",
        F.row_number()
        .over(window_sk_movies)
        .cast("long")
    )
    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
total_movies = df_dim_movies.count()

ids_naturais = (
    df_dim_movies
    .select("id_filme")
    .distinct()
    .count()
)

surrogate_keys = (
    df_dim_movies
    .select("sk_movie_id")
    .distinct()
    .count()
)

ids_nulos = (
    df_dim_movies
    .filter(F.col("id_filme").isNull())
    .count()
)

sk_nulas = (
    df_dim_movies
    .filter(F.col("sk_movie_id").isNull())
    .count()
)

print("=" * 60)
print("VALIDAÇÃO — GOLD.DIM_MOVIES")
print("=" * 60)

print(f"Total de filmes:          {total_movies}")
print(f"IDs naturais distintos:  {ids_naturais}")
print(f"Surrogate keys distintas: {surrogate_keys}")
print(f"IDs naturais nulos:       {ids_nulos}")
print(f"Surrogate keys nulas:     {sk_nulas}")

if (
    total_movies == ids_naturais
    and total_movies == surrogate_keys
    and ids_nulos == 0
    and sk_nulas == 0
):
    print(
        "[OK] Granularidade e chaves da "
        "dim_movies estão corretas."
    )
else:
    print("[ERRO] Existem inconsistências na dim_movies.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


VALIDAÇÃO — GOLD.DIM_MOVIES
Total de filmes:          97879
IDs naturais distintos:  97879
Surrogate keys distintas: 97879
IDs naturais nulos:       0
Surrogate keys nulas:     0
[OK] Granularidade e chaves da dim_movies estão corretas.


In [0]:
DIM_MOVIES_TARGET = (
    f"{CATALOG}.{GOLD_SCHEMA}.dim_movies"
)

(
    df_dim_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_MOVIES_TARGET)
)

print(
    f"[OK] Tabela criada/atualizada: "
    f"{DIM_MOVIES_TARGET}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] Tabela criada/atualizada: workspace.gold.dim_movies


In [0]:
df_dim_movies_validacao = spark.table(
    DIM_MOVIES_TARGET
)

print("=" * 60)
print("VALIDAÇÃO FINAL — GOLD.DIM_MOVIES")
print("=" * 60)

print(f"Tabela: {DIM_MOVIES_TARGET}")
print(
    f"Registros: "
    f"{df_dim_movies_validacao.count()}"
)

df_dim_movies_validacao.printSchema()

display(
    df_dim_movies_validacao
    .orderBy("sk_movie_id")
    .limit(20)
)

VALIDAÇÃO FINAL — GOLD.DIM_MOVIES
Tabela: workspace.gold.dim_movies
Registros: 97879
root
 |-- sk_movie_id: long (nullable = true)
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- ano_lancamento: integer (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = true)
 |-- sinopse: string (nullable = true)



sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,1000004,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
2,1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
3,1000007,KYLE BROWNRIGG: INTRODUCING LYLE,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
4,1000011,Worth Your Weight in Gold,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
5,1000014,On va manquer !,2018-05-15,2018,0,fr,Lançado,null
6,1000030,58 Hours: The Baby Jessica Story,2021-07-31,2021,0,es,Lançado,null
7,1000054,One Hundred Years and Hope,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope."
8,1000058,Homecoming,2023-07-12,2023,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances."
9,1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,2016,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!"
10,1000073,A Chance To Win,2023-05-03,2023,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever."


In [0]:
df_dim_genres_base = (
    df_generos
    .filter(
        F.col("genero").isNotNull()
        &
        (F.trim(F.col("genero")) != "")
    )
    .select(
        F.trim(
            F.col("genero")
        ).alias("nome_genero")
    )
    .distinct()
)

In [0]:
window_sk_genres = (
    Window.orderBy("nome_genero")
)

df_dim_genres = (
    df_dim_genres_base
    .withColumn(
        "sk_genre_id",
        F.row_number()
        .over(window_sk_genres)
        .cast("long")
    )
    .select(
        "sk_genre_id",
        "nome_genero"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_dim_people_base = (
    df_pessoas_empresas
    .filter(
        F.col("tipo").isin(
            "Ator",
            "Diretor",
            "Roteirista"
        )
    )
    .filter(
        F.col("nome").isNotNull()
        &
        (F.trim(F.col("nome")) != "")
    )
    .select(
        F.trim(
            F.col("nome")
        ).alias("nome_pessoa"),

        F.col("tipo").alias("tipo_pessoa")
    )
    .distinct()
)

In [0]:
window_sk_people = (
    Window.orderBy(
        "nome_pessoa",
        "tipo_pessoa"
    )
)

df_dim_people = (
    df_dim_people_base
    .withColumn(
        "sk_person_id",
        F.row_number()
        .over(window_sk_people)
        .cast("long")
    )
    .select(
        "sk_person_id",
        "nome_pessoa",
        "tipo_pessoa"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_dim_companies_base = (
    df_pessoas_empresas
    .filter(
        F.col("tipo") == "Produtora"
    )
    .filter(
        F.col("nome").isNotNull()
        &
        (F.trim(F.col("nome")) != "")
    )
    .select(
        F.trim(
            F.col("nome")
        ).alias("nome_produtora")
    )
    .distinct()
)

In [0]:
window_sk_companies = (
    Window.orderBy("nome_produtora")
)

df_dim_companies = (
    df_dim_companies_base
    .withColumn(
        "sk_company_id",
        F.row_number()
        .over(window_sk_companies)
        .cast("long")
    )
    .select(
        "sk_company_id",
        "nome_produtora"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
total_generos = df_dim_genres.count()

sk_generos = (
    df_dim_genres
    .select("sk_genre_id")
    .distinct()
    .count()
)

total_pessoas = df_dim_people.count()

sk_pessoas = (
    df_dim_people
    .select("sk_person_id")
    .distinct()
    .count()
)

total_produtoras = df_dim_companies.count()

sk_produtoras = (
    df_dim_companies
    .select("sk_company_id")
    .distinct()
    .count()
)

print("=" * 60)
print("VALIDAÇÃO — DIMENSÕES GOLD")
print("=" * 60)

print(
    f"dim_genres:    "
    f"{total_generos} registros | "
    f"{sk_generos} SKs distintas"
)

print(
    f"dim_people:    "
    f"{total_pessoas} registros | "
    f"{sk_pessoas} SKs distintas"
)

print(
    f"dim_companies: "
    f"{total_produtoras} registros | "
    f"{sk_produtoras} SKs distintas"
)

if (
    total_generos == sk_generos
    and total_pessoas == sk_pessoas
    and total_produtoras == sk_produtoras
):
    print(
        "[OK] Surrogate keys das dimensões "
        "são únicas."
    )
else:
    print(
        "[ERRO] Existem surrogate keys duplicadas."
    )


display(
    df_dim_people
    .groupBy("tipo_pessoa")
    .count()
    .orderBy("tipo_pessoa")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


VALIDAÇÃO — DIMENSÕES GOLD
dim_genres:    19 registros | 19 SKs distintas
dim_people:    420754 registros | 420754 SKs distintas
dim_companies: 45580 registros | 45580 SKs distintas
[OK] Surrogate keys das dimensões são únicas.


In [0]:
DIM_GENRES_TARGET = (
    f"{CATALOG}.{GOLD_SCHEMA}.dim_genres"
)

(
    df_dim_genres.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_GENRES_TARGET)
)

print(f"[OK] {DIM_GENRES_TARGET}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] workspace.gold.dim_genres


In [0]:
DIM_PEOPLE_TARGET = (
    f"{CATALOG}.{GOLD_SCHEMA}.dim_people"
)

(
    df_dim_people.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_PEOPLE_TARGET)
)

print(f"[OK] {DIM_PEOPLE_TARGET}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] workspace.gold.dim_people


In [0]:
DIM_COMPANIES_TARGET = (
    f"{CATALOG}.{GOLD_SCHEMA}.dim_companies"
)

(
    df_dim_companies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_COMPANIES_TARGET)
)

print(f"[OK] {DIM_COMPANIES_TARGET}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] workspace.gold.dim_companies


In [0]:
df_genres_gold = spark.table(
    DIM_GENRES_TARGET
)

df_people_gold = spark.table(
    DIM_PEOPLE_TARGET
)

df_companies_gold = spark.table(
    DIM_COMPANIES_TARGET
)

print("=" * 60)
print("VALIDAÇÃO FINAL — DIMENSÕES GOLD")
print("=" * 60)

print(
    f"dim_genres:    "
    f"{df_genres_gold.count()}"
)

print(
    f"dim_people:    "
    f"{df_people_gold.count()}"
)

print(
    f"dim_companies: "
    f"{df_companies_gold.count()}"
)

print("\nSCHEMA — DIM_GENRES")
df_genres_gold.printSchema()

print("\nSCHEMA — DIM_PEOPLE")
df_people_gold.printSchema()

print("\nSCHEMA — DIM_COMPANIES")
df_companies_gold.printSchema()

VALIDAÇÃO FINAL — DIMENSÕES GOLD
dim_genres:    19
dim_people:    420754
dim_companies: 45580

SCHEMA — DIM_GENRES
root
 |-- sk_genre_id: long (nullable = true)
 |-- nome_genero: string (nullable = true)


SCHEMA — DIM_PEOPLE
root
 |-- sk_person_id: long (nullable = true)
 |-- nome_pessoa: string (nullable = true)
 |-- tipo_pessoa: string (nullable = true)


SCHEMA — DIM_COMPANIES
root
 |-- sk_company_id: long (nullable = true)
 |-- nome_produtora: string (nullable = true)



In [0]:
df_bridge_movie_genre = (
    df_generos
    .alias("g")
    .join(
        df_dim_movies
        .select("sk_movie_id", "id_filme")
        .alias("m"),
        F.col("g.id_filme") == F.col("m.id_filme"),
        "inner"
    )
    .join(
        df_dim_genres
        .alias("dg"),
        F.col("g.genero") == F.col("dg.nome_genero"),
        "inner"
    )
    .select(
        F.col("m.sk_movie_id"),
        F.col("dg.sk_genre_id")
    )
    .dropDuplicates([
        "sk_movie_id",
        "sk_genre_id"
    ])
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_bridge_movie_person = (
    df_pessoas_empresas
    .filter(
        F.col("tipo").isin(
            "Ator",
            "Diretor",
            "Roteirista"
        )
    )
    .alias("p")
    .join(
        df_dim_movies
        .select("sk_movie_id", "id_filme")
        .alias("m"),
        F.col("p.id_filme") == F.col("m.id_filme"),
        "inner"
    )
    .join(
        df_dim_people.alias("dp"),
        (
            F.col("p.nome") == F.col("dp.nome_pessoa")
        )
        &
        (
            F.col("p.tipo") == F.col("dp.tipo_pessoa")
        ),
        "inner"
    )
    .select(
        F.col("m.sk_movie_id"),
        F.col("dp.sk_person_id")
    )
    .dropDuplicates([
        "sk_movie_id",
        "sk_person_id"
    ])
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_bridge_movie_company = (
    df_pessoas_empresas
    .filter(
        F.col("tipo") == "Produtora"
    )
    .alias("p")
    .join(
        df_dim_movies
        .select("sk_movie_id", "id_filme")
        .alias("m"),
        F.col("p.id_filme") == F.col("m.id_filme"),
        "inner"
    )
    .join(
        df_dim_companies.alias("dc"),
        F.col("p.nome") == F.col("dc.nome_produtora"),
        "inner"
    )
    .select(
        F.col("m.sk_movie_id"),
        F.col("dc.sk_company_id")
    )
    .dropDuplicates([
        "sk_movie_id",
        "sk_company_id"
    ])
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
def validar_bridge(df, nome, colunas_chave):
    total = df.count()

    distintos = (
        df
        .select(*colunas_chave)
        .distinct()
        .count()
    )

    condicao_nulos = None

    for coluna in colunas_chave:
        condicao = F.col(coluna).isNull()

        condicao_nulos = (
            condicao
            if condicao_nulos is None
            else condicao_nulos | condicao
        )

    nulos = (
        df
        .filter(condicao_nulos)
        .count()
    )

    print(f"\n{nome}")
    print(f"Total:      {total}")
    print(f"Distintos:  {distintos}")
    print(f"FKs nulas:  {nulos}")

    if total == distintos and nulos == 0:
        print("[OK]")
    else:
        print("[ERRO]")


print("=" * 60)
print("VALIDAÇÃO — BRIDGE TABLES")
print("=" * 60)

validar_bridge(
    df_bridge_movie_genre,
    "bridge_movie_genre",
    [
        "sk_movie_id",
        "sk_genre_id"
    ]
)

validar_bridge(
    df_bridge_movie_person,
    "bridge_movie_person",
    [
        "sk_movie_id",
        "sk_person_id"
    ]
)

validar_bridge(
    df_bridge_movie_company,
    "bridge_movie_company",
    [
        "sk_movie_id",
        "sk_company_id"
    ]
)

VALIDAÇÃO — BRIDGE TABLES


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(



bridge_movie_genre
Total:      131161
Distintos:  131161
FKs nulas:  0
[OK]

bridge_movie_person
Total:      785122
Distintos:  785122
FKs nulas:  0
[OK]

bridge_movie_company
Total:      118466
Distintos:  118466
FKs nulas:  0
[OK]


In [0]:
print("=" * 60)
print("COBERTURA — BRIDGES")
print("=" * 60)

print(
    "Gêneros Silver:",
    df_generos.count()
)

print(
    "Gêneros Gold:",
    df_bridge_movie_genre.count()
)

pessoas_silver = (
    df_pessoas_empresas
    .filter(
        F.col("tipo").isin(
            "Ator",
            "Diretor",
            "Roteirista"
        )
    )
    .count()
)

print(
    "Pessoas Silver:",
    pessoas_silver
)

print(
    "Pessoas Gold:",
    df_bridge_movie_person.count()
)

empresas_silver = (
    df_pessoas_empresas
    .filter(
        F.col("tipo") == "Produtora"
    )
    .count()
)

print(
    "Empresas Silver:",
    empresas_silver
)

print(
    "Empresas Gold:",
    df_bridge_movie_company.count()
)

COBERTURA — BRIDGES
Gêneros Silver: 132775


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gêneros Gold: 131161
Pessoas Silver: 795003
Pessoas Gold: 785122
Empresas Silver: 119819
Empresas Gold: 118466


In [0]:
BRIDGE_GENRE_TARGET = (
    f"{CATALOG}.{GOLD_SCHEMA}.bridge_movie_genre"
)

(
    df_bridge_movie_genre.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRIDGE_GENRE_TARGET)
)

print(f"[OK] {BRIDGE_GENRE_TARGET}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] workspace.gold.bridge_movie_genre


In [0]:
BRIDGE_PERSON_TARGET = (
    f"{CATALOG}.{GOLD_SCHEMA}.bridge_movie_person"
)

(
    df_bridge_movie_person.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRIDGE_PERSON_TARGET)
)

print(f"[OK] {BRIDGE_PERSON_TARGET}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] workspace.gold.bridge_movie_person


In [0]:
BRIDGE_COMPANY_TARGET = (
    f"{CATALOG}.{GOLD_SCHEMA}.bridge_movie_company"
)

(
    df_bridge_movie_company.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRIDGE_COMPANY_TARGET)
)

print(f"[OK] {BRIDGE_COMPANY_TARGET}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] workspace.gold.bridge_movie_company


In [0]:
df_bridge_genre_gold = spark.table(
    BRIDGE_GENRE_TARGET
)

df_bridge_person_gold = spark.table(
    BRIDGE_PERSON_TARGET
)

df_bridge_company_gold = spark.table(
    BRIDGE_COMPANY_TARGET
)

print("=" * 60)
print("VALIDAÇÃO FINAL — BRIDGE TABLES")
print("=" * 60)

print(
    f"bridge_movie_genre:   "
    f"{df_bridge_genre_gold.count()}"
)

print(
    f"bridge_movie_person:  "
    f"{df_bridge_person_gold.count()}"
)

print(
    f"bridge_movie_company: "
    f"{df_bridge_company_gold.count()}"
)

print("\nSCHEMA — BRIDGE_MOVIE_GENRE")
df_bridge_genre_gold.printSchema()

print("\nSCHEMA — BRIDGE_MOVIE_PERSON")
df_bridge_person_gold.printSchema()

print("\nSCHEMA — BRIDGE_MOVIE_COMPANY")
df_bridge_company_gold.printSchema()

VALIDAÇÃO FINAL — BRIDGE TABLES
bridge_movie_genre:   131161
bridge_movie_person:  785122
bridge_movie_company: 118466

SCHEMA — BRIDGE_MOVIE_GENRE
root
 |-- sk_movie_id: long (nullable = true)
 |-- sk_genre_id: long (nullable = true)


SCHEMA — BRIDGE_MOVIE_PERSON
root
 |-- sk_movie_id: long (nullable = true)
 |-- sk_person_id: long (nullable = true)


SCHEMA — BRIDGE_MOVIE_COMPANY
root
 |-- sk_movie_id: long (nullable = true)
 |-- sk_company_id: long (nullable = true)



In [0]:
df_reviews_agregado = (
    df_avaliacoes
    .groupBy("id_filme")
    .agg(
        F.count("*")
        .cast("int")
        .alias("qtd_avaliacoes_usuarios"),

        F.round(
            F.avg("nota_usuario"),
            2
        )
        .cast("double")
        .alias("nota_media_usuarios")
    )
)

In [0]:
df_dim_reviews_base = (
    df_reviews_agregado
    .alias("r")
    .join(
        df_dim_movies
        .select(
            "sk_movie_id",
            "id_filme"
        )
        .alias("m"),
        F.col("r.id_filme") == F.col("m.id_filme"),
        "inner"
    )
    .select(
        F.col("m.sk_movie_id"),
        F.col("r.qtd_avaliacoes_usuarios"),
        F.col("r.nota_media_usuarios")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
window_sk_reviews = (
    Window.orderBy("sk_movie_id")
)

df_dim_reviews = (
    df_dim_reviews_base
    .withColumn(
        "sk_review_id",
        F.row_number()
        .over(window_sk_reviews)
        .cast("long")
    )
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
total_reviews_gold = df_dim_reviews.count()

sk_reviews_distintas = (
    df_dim_reviews
    .select("sk_review_id")
    .distinct()
    .count()
)

filmes_reviews_distintos = (
    df_dim_reviews
    .select("sk_movie_id")
    .distinct()
    .count()
)

fks_nulas = (
    df_dim_reviews
    .filter(
        F.col("sk_movie_id").isNull()
    )
    .count()
)

quantidades_invalidas = (
    df_dim_reviews
    .filter(
        F.col("qtd_avaliacoes_usuarios") <= 0
    )
    .count()
)

notas_invalidas = (
    df_dim_reviews
    .filter(
        F.col("nota_media_usuarios").isNotNull()
        &
        (
            (F.col("nota_media_usuarios") < 0)
            |
            (F.col("nota_media_usuarios") > 10)
        )
    )
    .count()
)

print("=" * 60)
print("VALIDAÇÃO — GOLD.DIM_REVIEWS")
print("=" * 60)

print(f"Total de filmes avaliados: {total_reviews_gold}")
print(f"SKs distintas:             {sk_reviews_distintas}")
print(f"Filmes distintos:          {filmes_reviews_distintos}")
print(f"FKs nulas:                 {fks_nulas}")
print(f"Quantidades inválidas:     {quantidades_invalidas}")
print(f"Notas médias inválidas:    {notas_invalidas}")

if (
    total_reviews_gold == sk_reviews_distintas
    and total_reviews_gold == filmes_reviews_distintos
    and fks_nulas == 0
    and quantidades_invalidas == 0
    and notas_invalidas == 0
):
    print("[OK] dim_reviews válida.")
else:
    print("[ERRO] Existem inconsistências na dim_reviews.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


VALIDAÇÃO — GOLD.DIM_REVIEWS
Total de filmes avaliados: 27303
SKs distintas:             27303
Filmes distintos:          27303
FKs nulas:                 0
Quantidades inválidas:     0
Notas médias inválidas:    0
[OK] dim_reviews válida.


In [0]:
avaliacoes_silver = df_avaliacoes.count()

avaliacoes_gold = (
    df_dim_reviews
    .agg(
        F.sum("qtd_avaliacoes_usuarios")
        .alias("total")
    )
    .first()["total"]
)

print("=" * 60)
print("COBERTURA — AVALIAÇÕES")
print("=" * 60)

print(f"Avaliações Silver: {avaliacoes_silver}")
print(f"Avaliações Gold:   {avaliacoes_gold}")
print(
    f"Não associadas à dim_movies: "
    f"{avaliacoes_silver - avaliacoes_gold}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


COBERTURA — AVALIAÇÕES
Avaliações Silver: 32412
Avaliações Gold:   31993
Não associadas à dim_movies: 419


In [0]:
DIM_REVIEWS_TARGET = (
    f"{CATALOG}.{GOLD_SCHEMA}.dim_reviews"
)

(
    df_dim_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_REVIEWS_TARGET)
)

print(
    f"[OK] Tabela criada/atualizada: "
    f"{DIM_REVIEWS_TARGET}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] Tabela criada/atualizada: workspace.gold.dim_reviews


In [0]:
df_reviews_gold = spark.table(
    DIM_REVIEWS_TARGET
)

print("=" * 60)
print("VALIDAÇÃO FINAL — GOLD.DIM_REVIEWS")
print("=" * 60)

print(f"Tabela: {DIM_REVIEWS_TARGET}")
print(f"Registros: {df_reviews_gold.count()}")

df_reviews_gold.printSchema()

display(
    df_reviews_gold
    .orderBy(
        F.col("qtd_avaliacoes_usuarios").desc()
    )
    .limit(20)
)

VALIDAÇÃO FINAL — GOLD.DIM_REVIEWS
Tabela: workspace.gold.dim_reviews
Registros: 27303
root
 |-- sk_review_id: long (nullable = true)
 |-- sk_movie_id: long (nullable = true)
 |-- qtd_avaliacoes_usuarios: integer (nullable = true)
 |-- nota_media_usuarios: double (nullable = true)



sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
4212,14792,5,4.73
8571,30308,4,6.1
1468,5293,4,4.38
9718,34497,4,4.53
5947,20949,4,7.07
7382,26078,4,3.88
2067,7379,4,6.23
513,1834,4,5.27
9167,32536,4,2.53
5258,18550,4,4.9


In [0]:
df_movies_lancados = (
    df_dim_movies
    .filter(
        F.col("status_filme") == "Lançado"
    )
)

print("=" * 60)
print("FILMES LANÇADOS")
print("=" * 60)

print(
    f"Total dim_movies: "
    f"{df_dim_movies.count()}"
)

print(
    f"Filmes lançados: "
    f"{df_movies_lancados.count()}"
)

FILMES LANÇADOS


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total dim_movies: 97879
Filmes lançados: 96463


In [0]:
df_fact_movies_performance = (
    df_movies_lancados
    .select(
        "sk_movie_id",
        "id_filme"
    )
    .alias("m")

    .join(
        df_financeiro.alias("f"),
        F.col("m.id_filme") == F.col("f.id_filme"),
        "left"
    )

    .join(
        df_metricas.alias("e"),
        F.col("m.id_filme") == F.col("e.id_filme"),
        "left"
    )

    .select(
        F.col("m.sk_movie_id"),

        F.col("f.orcamento_usd")
        .cast("decimal(18,2)")
        .alias("orcamento_usd"),

        F.col("f.receita_usd")
        .cast("decimal(18,2)")
        .alias("receita_usd"),

        F.col("f.lucro_usd")
        .cast("decimal(18,2)")
        .alias("lucro_usd"),

        F.col("f.orcamento_brl")
        .cast("decimal(18,2)")
        .alias("orcamento_brl"),

        F.col("f.receita_brl")
        .cast("decimal(18,2)")
        .alias("receita_brl"),

        F.col("f.lucro_brl")
        .cast("decimal(18,2)")
        .alias("lucro_brl"),

        F.col("e.popularidade")
        .cast("double")
        .alias("popularidade"),

        F.col("e.nota_media_tmdb")
        .cast("double")
        .alias("nota_media_tmdb"),

        F.col("e.qtd_votos_tmdb")
        .cast("int")
        .alias("qtd_votos_tmdb"),

        F.col("e.nota_media_imdb")
        .cast("double")
        .alias("nota_media_imdb"),

        F.col("e.qtd_votos_imdb")
        .cast("int")
        .alias("qtd_votos_imdb")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
total_fact = (
    df_fact_movies_performance.count()
)

filmes_distintos_fact = (
    df_fact_movies_performance
    .select("sk_movie_id")
    .distinct()
    .count()
)

filmes_lancados = (
    df_movies_lancados.count()
)

sk_nulas = (
    df_fact_movies_performance
    .filter(
        F.col("sk_movie_id").isNull()
    )
    .count()
)

print("=" * 60)
print(
    "VALIDAÇÃO — "
    "FACT_MOVIES_PERFORMANCE"
)
print("=" * 60)

print(
    f"Filmes lançados:       "
    f"{filmes_lancados}"
)

print(
    f"Registros na fact:     "
    f"{total_fact}"
)

print(
    f"SKs distintas:         "
    f"{filmes_distintos_fact}"
)

print(
    f"SKs nulas:             "
    f"{sk_nulas}"
)

if (
    total_fact == filmes_lancados
    and total_fact == filmes_distintos_fact
    and sk_nulas == 0
):
    print(
        "[OK] Granularidade 1 filme = "
        "1 registro preservada."
    )
else:
    print(
        "[ERRO] A granularidade da fact "
        "foi comprometida."
    )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


VALIDAÇÃO — FACT_MOVIES_PERFORMANCE
Filmes lançados:       96463
Registros na fact:     96463
SKs distintas:         96463
SKs nulas:             0
[OK] Granularidade 1 filme = 1 registro preservada.


In [0]:
status_fact = (
    df_fact_movies_performance
    .alias("f")
    .join(
        df_dim_movies
        .select(
            "sk_movie_id",
            "status_filme"
        )
        .alias("m"),
        "sk_movie_id",
        "inner"
    )
    .groupBy("status_filme")
    .count()
)

display(status_fact)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


status_filme,count
Lançado,96463


In [0]:
print("=" * 60)
print("COBERTURA — FACT")
print("=" * 60)

df_fact_movies_performance.select(
    F.count("*")
    .alias("total_filmes"),

    F.count("orcamento_usd")
    .alias("com_orcamento"),

    F.count("receita_usd")
    .alias("com_receita"),

    F.count("popularidade")
    .alias("com_popularidade"),

    F.count("nota_media_tmdb")
    .alias("com_nota_tmdb"),

    F.count("nota_media_imdb")
    .alias("com_nota_imdb")
).show()    

COBERTURA — FACT


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------------+-------------+-----------+----------------+-------------+-------------+
|total_filmes|com_orcamento|com_receita|com_popularidade|com_nota_tmdb|com_nota_imdb|
+------------+-------------+-----------+----------------+-------------+-------------+
|       96463|         7951|       3256|           92267|        92880|        83480|
+------------+-------------+-----------+----------------+-------------+-------------+



In [0]:
metricas_invalidas_fact = (
    df_fact_movies_performance
    .filter(
        (F.col("popularidade") < 0)
        |
        (
            F.col("nota_media_tmdb").isNotNull()
            &
            ~F.col("nota_media_tmdb").between(0, 10)
        )
        |
        (
            F.col("qtd_votos_tmdb").isNotNull()
            &
            (F.col("qtd_votos_tmdb") < 0)
        )
        |
        (
            F.col("nota_media_imdb").isNotNull()
            &
            ~F.col("nota_media_imdb").between(0, 10)
        )
        |
        (
            F.col("qtd_votos_imdb").isNotNull()
            &
            (F.col("qtd_votos_imdb") < 0)
        )
    )
    .count()
)

print(
    f"Métricas inválidas na fact: "
    f"{metricas_invalidas_fact}"
)

if metricas_invalidas_fact == 0:
    print("[OK] Métricas válidas.")
else:
    print("[ERRO] Existem métricas inválidas.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Métricas inválidas na fact: 0
[OK] Métricas válidas.


In [0]:
FACT_MOVIES_TARGET = (
    f"{CATALOG}.{GOLD_SCHEMA}."
    "fact_movies_performance"
)

(
    df_fact_movies_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_MOVIES_TARGET)
)

print(
    f"[OK] Tabela criada/atualizada: "
    f"{FACT_MOVIES_TARGET}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] Tabela criada/atualizada: workspace.gold.fact_movies_performance


In [0]:
df_fact_gold = spark.table(
    FACT_MOVIES_TARGET
)

print("=" * 60)
print(
    "VALIDAÇÃO FINAL — "
    "GOLD.FACT_MOVIES_PERFORMANCE"
)
print("=" * 60)

print(f"Tabela: {FACT_MOVIES_TARGET}")
print(f"Registros: {df_fact_gold.count()}")

df_fact_gold.printSchema()

display(
    df_fact_gold.limit(20)
)

VALIDAÇÃO FINAL — GOLD.FACT_MOVIES_PERFORMANCE
Tabela: workspace.gold.fact_movies_performance
Registros: 96463
root
 |-- sk_movie_id: long (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)



sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1,null,null,null,null,null,null,1.132,0.0,0,6.8,27
2,null,null,null,null,null,null,0.6,0.0,0,null,40
3,null,null,null,null,null,null,0.6,0.0,0,4.7,10
4,null,null,null,null,null,null,1.169,0.0,0,4.8,16
5,null,null,null,null,null,null,0.6,0.0,0,7.2,15
6,null,null,null,null,null,null,0.615,0.0,0,null,25
7,null,null,null,null,null,null,0.6,0.0,0,5.9,10
8,4700000.00,null,null,24237430.00,null,null,1.489,6.75,6,6.2,382
9,null,null,null,null,null,null,0.6,0.0,0,7.7,25
10,6000000.00,null,null,30941400.00,null,null,13.212,6.8,15,null,236


In [0]:
df_atores_contexto = (
    df_bridge_person_gold
    .alias("b")
    .join(
        df_people_gold
        .filter(F.col("tipo_pessoa") == "Ator")
        .alias("p"),
        F.col("b.sk_person_id")
        == F.col("p.sk_person_id"),
        "inner"
    )
    .groupBy(
        F.col("b.sk_movie_id")
    )
    .agg(
        F.concat_ws(
            ", ",
            F.sort_array(
                F.collect_set("p.nome_pessoa")
            )
        ).alias("atores")
    )
)

In [0]:
df_diretores_contexto = (
    df_bridge_person_gold
    .alias("b")
    .join(
        df_people_gold
        .filter(
            F.col("tipo_pessoa") == "Diretor"
        )
        .alias("p"),
        F.col("b.sk_person_id")
        == F.col("p.sk_person_id"),
        "inner"
    )
    .groupBy(
        F.col("b.sk_movie_id")
    )
    .agg(
        F.concat_ws(
            ", ",
            F.sort_array(
                F.collect_set("p.nome_pessoa")
            )
        ).alias("diretores")
    )
)

In [0]:
df_genai_base = (
    df_dim_movies
    .alias("m")
    .join(
        df_fact_gold.alias("f"),
        F.col("m.sk_movie_id")
        == F.col("f.sk_movie_id"),
        "inner"
    )
    .join(
        df_atores_contexto.alias("a"),
        F.col("m.sk_movie_id")
        == F.col("a.sk_movie_id"),
        "left"
    )
    .join(
        df_diretores_contexto.alias("d"),
        F.col("m.sk_movie_id")
        == F.col("d.sk_movie_id"),
        "left"
    )
    .select(
        F.col("m.id_filme").alias("movie_id"),
        F.col("m.titulo").alias("title"),
        F.col("m.ano_lancamento"),
        F.col("m.sinopse"),
        F.col("f.receita_usd"),
        F.col("f.orcamento_usd"),
        F.col("a.atores"),
        F.col("d.diretores")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_genai_context = (
    df_genai_base
    .withColumn(
        "llm_context_document",
        F.concat(
            F.lit("O filme "),
            F.coalesce(
                F.col("title"),
                F.lit("Não Informado")
            ),

            F.lit(", lançado no ano de "),
            F.coalesce(
                F.col("ano_lancamento")
                .cast("string"),
                F.lit("Não Informado")
            ),

            F.lit(", faturou "),
            F.coalesce(
                F.concat(
                    F.lit("US$ "),
                    F.format_number(
                        F.col("receita_usd"),
                        2
                    )
                ),
                F.lit("valor não informado")
            ),

            F.lit(" e teve um custo de "),
            F.coalesce(
                F.concat(
                    F.lit("US$ "),
                    F.format_number(
                        F.col("orcamento_usd"),
                        2
                    )
                ),
                F.lit("valor não informado")
            ),

            F.lit(". Estrelado por "),
            F.coalesce(
                F.col("atores"),
                F.lit("elenco não informado")
            ),

            F.lit(" e dirigido por "),
            F.coalesce(
                F.col("diretores"),
                F.lit("diretor não informado")
            ),

            F.lit(
                ", o filme possui a seguinte sinopse: "
            ),

            F.coalesce(
                F.col("sinopse"),
                F.lit("Sinopse não informada")
            ),

            F.lit(".")
        )
    )
    .select(
        "movie_id",
        "title",
        "llm_context_document"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
total_contextos = (
    df_genai_context.count()
)

movie_ids_distintos = (
    df_genai_context
    .select("movie_id")
    .distinct()
    .count()
)

movie_ids_nulos = (
    df_genai_context
    .filter(F.col("movie_id").isNull())
    .count()
)

documentos_nulos = (
    df_genai_context
    .filter(
        F.col("llm_context_document").isNull()
    )
    .count()
)

documentos_vazios = (
    df_genai_context
    .filter(
        F.trim(
            F.col("llm_context_document")
        ) == ""
    )
    .count()
)

print("=" * 60)
print(
    "VALIDAÇÃO — "
    "GOLD_GENAI_MOVIES_CONTEXT"
)
print("=" * 60)

print(f"Total de contextos:       {total_contextos}")
print(f"Movie IDs distintos:      {movie_ids_distintos}")
print(f"Movie IDs nulos:          {movie_ids_nulos}")
print(f"Documentos nulos:         {documentos_nulos}")
print(f"Documentos vazios:        {documentos_vazios}")

if (
    total_contextos == movie_ids_distintos
    and movie_ids_nulos == 0
    and documentos_nulos == 0
    and documentos_vazios == 0
):
    print(
        "[OK] Contextos GenAI válidos."
    )
else:
    print(
        "[ERRO] Existem inconsistências."
    )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


VALIDAÇÃO — GOLD_GENAI_MOVIES_CONTEXT
Total de contextos:       96463
Movie IDs distintos:      96463
Movie IDs nulos:          0
Documentos nulos:         0
Documentos vazios:        0
[OK] Contextos GenAI válidos.


In [0]:
display(
    df_genai_context
    .select(
        "movie_id",
        "title",
        "llm_context_document"
    )
    .limit(20)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


movie_id,title,llm_context_document
1000004,Purple Beatz,"O filme Purple Beatz, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por Aron von Andrian, Erika Alexander, Izzy Jones, Steven Michael-O’Hara, Tedroy Newell e dirigido por Lola Atkins, o filme possui a seguinte sinopse: Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.."
1000005,Aisha Brown: The First Black Woman Ever,"O filme Aisha Brown: The First Black Woman Ever, lançado no ano de 2020, faturou valor não informado e teve um custo de valor não informado. Estrelado por Aisha Brown e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.."
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,"O filme KYLE BROWNRIGG: INTRODUCING LYLE, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por Kyle Brownrigg e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.."
1000011,Worth Your Weight in Gold,"O filme Worth Your Weight in Gold, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por Inês Pronto, Isabel Abreu, João Pedro Bénard, Marcello Urgeghe e dirigido por Sandro Aguilar, o filme possui a seguinte sinopse: Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.."
1000014,On va manquer !,"O filme On va manquer !, lançado no ano de 2018, faturou valor não informado e teve um custo de valor não informado. Estrelado por Biyouna, Claire Chust, Gabrielle Cohen, Maxime Pambet, Mouss Zouheyri, Soulayman Rkiba e dirigido por Sabrina Ouazani, o filme possui a seguinte sinopse: Sinopse não informada."
1000030,58 Hours: The Baby Jessica Story,"O filme 58 Hours: The Baby Jessica Story, lançado no ano de 2021, faturou valor não informado e teve um custo de valor não informado. Estrelado por elenco não informado e dirigido por Mark Bone, o filme possui a seguinte sinopse: Sinopse não informada."
1000054,One Hundred Years and Hope,"O filme One Hundred Years and Hope, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por elenco não informado e dirigido por Takashi Nishihara, o filme possui a seguinte sinopse: In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope.."
1000058,Homecoming,"O filme Homecoming, lançado no ano de 2023, faturou valor não informado e teve um custo de US$ 4,700,000.00. Estrelado por A

In [0]:
GENAI_TARGET = (
    f"{CATALOG}.{GOLD_SCHEMA}."
    "gold_genai_movies_context"
)

(
    df_genai_context.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GENAI_TARGET)
)

print(
    f"[OK] Tabela criada/atualizada: "
    f"{GENAI_TARGET}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] Tabela criada/atualizada: workspace.gold.gold_genai_movies_context


In [0]:
df_genai_gold = spark.table(
    GENAI_TARGET
)

print("=" * 60)
print(
    "VALIDAÇÃO FINAL — "
    "GOLD_GENAI_MOVIES_CONTEXT"
)
print("=" * 60)

print(f"Tabela: {GENAI_TARGET}")
print(f"Registros: {df_genai_gold.count()}")

df_genai_gold.printSchema()

display(
    df_genai_gold.limit(10)
)

VALIDAÇÃO FINAL — GOLD_GENAI_MOVIES_CONTEXT
Tabela: workspace.gold.gold_genai_movies_context
Registros: 96463
root
 |-- movie_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- llm_context_document: string (nullable = true)



movie_id,title,llm_context_document
1000004,Purple Beatz,"O filme Purple Beatz, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por Aron von Andrian, Erika Alexander, Izzy Jones, Steven Michael-O’Hara, Tedroy Newell e dirigido por Lola Atkins, o filme possui a seguinte sinopse: Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.."
1000005,Aisha Brown: The First Black Woman Ever,"O filme Aisha Brown: The First Black Woman Ever, lançado no ano de 2020, faturou valor não informado e teve um custo de valor não informado. Estrelado por Aisha Brown e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.."
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,"O filme KYLE BROWNRIGG: INTRODUCING LYLE, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por Kyle Brownrigg e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.."
1000011,Worth Your Weight in Gold,"O filme Worth Your Weight in Gold, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por Inês Pronto, Isabel Abreu, João Pedro Bénard, Marcello Urgeghe e dirigido por Sandro Aguilar, o filme possui a seguinte sinopse: Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.."
1000014,On va manquer !,"O filme On va manquer !, lançado no ano de 2018, faturou valor não informado e teve um custo de valor não informado. Estrelado por Biyouna, Claire Chust, Gabrielle Cohen, Maxime Pambet, Mouss Zouheyri, Soulayman Rkiba e dirigido por Sabrina Ouazani, o filme possui a seguinte sinopse: Sinopse não informada."
1000030,58 Hours: The Baby Jessica Story,"O filme 58 Hours: The Baby Jessica Story, lançado no ano de 2021, faturou valor não informado e teve um custo de valor não informado. Estrelado por elenco não informado e dirigido por Mark Bone, o filme possui a seguinte sinopse: Sinopse não informada."
1000054,One Hundred Years and Hope,"O filme One Hundred Years and Hope, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por elenco não informado e dirigido por Takashi Nishihara, o filme possui a seguinte sinopse: In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope.."
1000058,Homecoming,"O filme Homecoming, lançado no ano de 2023, faturou valor não informado e teve um custo de US$ 4,700,000.00. Estrelado por A

In [0]:
df_fact_gold.createOrReplaceTempView("fact_movies_performance")
df_dim_movies.createOrReplaceTempView("dim_movies")
df_genres_gold.createOrReplaceTempView("dim_genres")
df_people_gold.createOrReplaceTempView("dim_people")
df_companies_gold.createOrReplaceTempView("dim_companies")

df_bridge_genre_gold.createOrReplaceTempView("bridge_movie_genre")
df_bridge_person_gold.createOrReplaceTempView("bridge_movie_person")
df_bridge_company_gold.createOrReplaceTempView("bridge_movie_company")

print("[OK] Views temporárias criadas.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] Views temporárias criadas.


In [0]:
# Receita Total BRL
query_receita_total = """
SELECT
    ROUND(
        SUM(receita_brl),
        2
    ) AS receita_total_brl
FROM fact_movies_performance
"""

display(
    spark.sql(query_receita_total)
)

receita_total_brl
834732290730.20


In [0]:
# Top 5 filmes por popularidade
query_top5_popularidade = """
SELECT
    m.titulo,
    f.popularidade
FROM fact_movies_performance f
INNER JOIN dim_movies m
    ON f.sk_movie_id = m.sk_movie_id
WHERE f.popularidade IS NOT NULL
ORDER BY
    f.popularidade DESC,
    m.titulo ASC
LIMIT 5
"""

display(
    spark.sql(query_top5_popularidade)
)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
retribution,1547.22


In [0]:
# Quantidade de filmes por gênero
query_filmes_genero = """
SELECT
    g.nome_genero,
    COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
FROM bridge_movie_genre b
INNER JOIN dim_genres g
    ON b.sk_genre_id = g.sk_genre_id
GROUP BY
    g.sk_genre_id,
    g.nome_genero
ORDER BY
    qtd_filmes DESC,
    g.nome_genero ASC
"""

display(
    spark.sql(query_filmes_genero)
)

nome_genero,qtd_filmes
Drama,30457
Documentary,18612
Comedy,17348
Thriller,9424
Horror,9163
Romance,6996
Action,5541
Crime,4320
Animation,4159
TV Movie,3676


In [0]:
# Top 10 filmes por receita
query_top10_receita = """
WITH ranking_receita AS (
    SELECT
        m.titulo,
        f.receita_usd,
        f.receita_brl,
        RANK() OVER (
            ORDER BY f.receita_usd DESC
        ) AS posicao
    FROM fact_movies_performance f
    INNER JOIN dim_movies m
        ON f.sk_movie_id = m.sk_movie_id
    WHERE f.receita_usd IS NOT NULL
)

SELECT
    posicao,
    titulo,
    receita_usd,
    receita_brl
FROM ranking_receita
WHERE posicao <= 10
ORDER BY
    posicao,
    titulo
"""

display(
    spark.sql(query_top10_receita)
)

posicao,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,14439320000.00
2,Avatar: The Way of Water,2320250281.00,11965298674.09
3,AVENGERS: INFINITY WAR,2052415039.00,10584099114.62
4,spider-man: no way home,1921847111.00,9910773366.72
5,The Lion King,1663075401.00,8576313535.42
6,Top Gun: Maverick,1488732821.00,7677246284.61
7,Barbie,1428545028.00,7366863854.89
8,The Super Mario Bros. Movie,1355725263.00,6991339608.76
9,Black Panther,1349926083.00,6961433817.42
10,Star Wars: The Last Jedi,1332698830.00,6872594596.43


In [0]:
# Data para referência dos atores que participaram de filmes nos últimos 2 anos
query_data_referencia = """
SELECT
    MAX(data_lancamento) AS data_referencia
FROM dim_movies
WHERE status_filme = 'Lançado'
  AND data_lancamento IS NOT NULL
  AND data_lancamento <= CURRENT_DATE()
"""

display(
    spark.sql(query_data_referencia)
)

data_referencia
2026-02-19


In [0]:
# Atores com mais participações nos últimos 2 anos
query_ator_2_anos = """
WITH referencia AS (
    SELECT
        MAX(data_lancamento) AS data_maxima
    FROM dim_movies
    WHERE status_filme = 'Lançado'
      AND data_lancamento IS NOT NULL
      AND data_lancamento <= CURRENT_DATE()
),

participacoes AS (
    SELECT
        p.nome_pessoa,
        COUNT(DISTINCT m.sk_movie_id) AS qtd_filmes
    FROM bridge_movie_person b

    INNER JOIN dim_people p
        ON b.sk_person_id = p.sk_person_id

    INNER JOIN dim_movies m
        ON b.sk_movie_id = m.sk_movie_id

    CROSS JOIN referencia r

    WHERE p.tipo_pessoa = 'Ator'

      AND m.status_filme = 'Lançado'

      AND m.data_lancamento
          BETWEEN ADD_MONTHS(
              r.data_maxima,
              -24
          )
          AND r.data_maxima

    GROUP BY
        p.nome_pessoa
)

SELECT
    nome_pessoa,
    qtd_filmes
FROM participacoes
ORDER BY
    qtd_filmes DESC,
    nome_pessoa ASC
LIMIT 1
"""

display(
    spark.sql(query_ator_2_anos)
)

nome_pessoa,qtd_filmes
Kevin Hart,64


In [0]:
# Produtoras com mais lucro nos últimos 5 anos
query_produtora_5_anos = """
WITH referencia AS (
    SELECT
        MAX(data_lancamento) AS data_maxima
    FROM dim_movies
    WHERE status_filme = 'Lançado'
      AND data_lancamento IS NOT NULL
      AND data_lancamento <= CURRENT_DATE()
),

lucro_produtoras AS (
    SELECT
        c.nome_produtora,

        SUM(
            f.lucro_usd
        ) AS lucro_total_usd,

        SUM(
            f.lucro_brl
        ) AS lucro_total_brl

    FROM bridge_movie_company b

    INNER JOIN dim_companies c
        ON b.sk_company_id
         = c.sk_company_id

    INNER JOIN dim_movies m
        ON b.sk_movie_id
         = m.sk_movie_id

    INNER JOIN fact_movies_performance f
        ON b.sk_movie_id
         = f.sk_movie_id

    CROSS JOIN referencia r

    WHERE m.status_filme = 'Lançado'

      AND m.data_lancamento
          BETWEEN ADD_MONTHS(
              r.data_maxima,
              -60
          )
          AND r.data_maxima

      AND f.lucro_usd IS NOT NULL

    GROUP BY
        c.nome_produtora
)

SELECT
    nome_produtora,

    ROUND(
        lucro_total_usd,
        2
    ) AS lucro_total_usd,

    ROUND(
        lucro_total_brl,
        2
    ) AS lucro_total_brl

FROM lucro_produtoras

ORDER BY
    lucro_total_usd DESC,
    nome_produtora ASC

LIMIT 1
"""

display(
    spark.sql(query_produtora_5_anos)
)

nome_produtora,lucro_total_usd,lucro_total_brl
Universal Pictures,5772329679.00,29767326921.64
